# Consommer vs exposer le MCP — les deux sens du fil

*Recycler la documentation du projet LivresAgités (en sommeil) vers le dépôt
pédagogique. Ce notebook rend exécutable la leçon du **Parcours 0** du dossier
[`AI-Engine-WordPress`](./livresagites-parcours.md) : AI-Engine est un des rares
produits à jouer **les deux rôles** du protocole MCP — il *expose* WordPress
comme serveur MCP (un agent externe appelle ses outils), et il *consomme* des
serveurs MCP externes (module Orchestration). La confusion entre les deux est
le piège le plus fréquent du dossier.*

> **Thèse.** Les deux sens du protocole ne se configurent pas au même endroit,
> ne servent pas le même public, et — c'est ce que ce notebook mesure — ne
> *devraient pas* se chevaucher fonctionnellement. Quand un outil exposé et un
> outil consommé font la même chose, l'agent qui les a tous les deux doit
> choisir, et le double-écrit devient un risque.

Ce notebook ne dépend d'aucun service : pas de réseau, pas de modèle, pas de
clé. Un **mini-serveur MCP** et un **mini-client MCP** sont construits en
mémoire, sur le même fixture synthétique « Maison Valmont ». Ce qui est
enseigné est la *structure* du chevauchement, pas une mesure sur une instance
réelle.


## 1. Le protocole MCP en une phrase

**Model Context Protocol** est un protocole client-serveur par lequel un agent
(LLM) découvre et appelle des **outils** exposés par une application. Le
serveur publie un *catalogue* (liste d'outils avec leur schéma) ; le client
(appelé *host* ou *agent*) choisit un outil, fournit des arguments, reçoit un
résultat. Le modèle ne « sait » rien faire lui-même : tout passe par les
outils qu'on lui expose.

Deux rôles, donc : **serveur** (j'expose mes outils) et **client** (j'appelle
les outils des autres). Une même application peut jouer les deux. AI-Engine le
fait ; c'est rare. La plupart des produits ne jouent qu'un rôle.


## 2. Deux catalogues synthétiques — un exposé, un consommé

On monte la « Maison Valmont » avec son catalogue d'outils **exposés**
(`valmont_*`, ceux qu'un agent externe peut appeler sur le site), et un
catalogue d'outils **consommés** depuis un serveur MCP externe hypothétique
(un service d'enrichissement de fiches livre, `bookgraph_*`). Les deux sont
des dicts en mémoire — aucun réseau.


In [1]:
# Aucune dependance externe. Tout est en memoire.

# --- Catalogue EXPOSE par la Maison Valmont (serveur MCP) ---
# Un agent externe (Claude Code, ChatGPT...) appelle ces outils sur Valmont.
catalogue_expose = {
    "valmont_get_manuscripts":      {"verbe": "lire",       "cible": "manuscrit"},
    "valmont_submit_manuscript":    {"verbe": "soumettre",  "cible": "manuscrit"},
    "valmont_assign_reviewer":      {"verbe": "assigner",   "cible": "lecteur"},
    "valmont_get_book":             {"verbe": "lire",       "cible": "livre"},
    "valmont_search_catalog":       {"verbe": "chercher",   "cible": "livre"},
    "valmont_update_book_meta":     {"verbe": "modifier",   "cible": "livre"},
    "valmont_get_review":           {"verbe": "lire",       "cible": "compte-rendu"},
}

# --- Catalogue CONSOMME depuis un serveur MCP externe (bookgraph) ---
# Valmont, via son module Orchestration, appelle ces outils chez bookgraph.
catalogue_consomme = {
    "bookgraph_get_book_info":      {"verbe": "lire",       "cible": "livre"},
    "bookgraph_search_books":       {"verbe": "chercher",   "cible": "livre"},
    "bookgraph_update_book":        {"verbe": "modifier",   "cible": "livre"},
    "bookgraph_enrich_book":        {"verbe": "enrichir",   "cible": "livre"},
    "bookgraph_get_author":         {"verbe": "lire",       "cible": "auteur"},
    "bookgraph_link_similar":       {"verbe": "lier",       "cible": "livre"},
    "bookgraph_submit_review":      {"verbe": "soumettre",  "cible": "compte-rendu"},
}

print("Catalogue EXPOSE (Valmont sert)  : " + str(len(catalogue_expose)) + " outils")
print("Catalogue CONSOMME (Valmont appelle) : " + str(len(catalogue_consomme)) + " outils")


Catalogue EXPOSE (Valmont sert)  : 7 outils
Catalogue CONSOMME (Valmont appelle) : 7 outils


### Les deux rôles, sur le même site

| | Exposé par Valmont | Consommé par Valmont |
|---|---|---|
| **Sens** | agent externe → Valmont | Valmont → serveur externe |
| **Préfixe** | `valmont_*` | `bookgraph_*` |
| **Public** | Claude Code, ChatGPT, OpenClaw… | Le chatbot interne de Valmont |
| **Configuré dans** | AI-Engine : serveur MCP (Bearer token) | AI-Engine : module Orchestration |

**La confusion fréquente** : un administrateur qui veut « connecter un outil
MCP » ne sait pas s'il doit l'ajouter côté serveur (exposer un nouvel outil
Valmont aux agents externes) ou côté client (consommer un outil externe depuis
Valmont). La réponse dépend du **sens du fil** : qui appelle qui ?


## 3. Le mini-serveur et le mini-client

Deux fonctions simulent les deux rôles, en mémoire. Le serveur répond à un
appel d'outil ; le client appelle un outil chez un serveur externe. Le
comportement réel (réseau, JSON-RPC, authentification) est remplacé par des
dicts — la *structure* du protocole est ce qui compte ici.


In [2]:
def servir(catalogue_serveur, nom_outil, arguments):
    """Cote SERVEUR : un agent externe appelle un outil expose par Valmont."""
    if nom_outil not in catalogue_serveur:
        return {"erreur": "outil inconnu: " + nom_outil}
    spec = catalogue_serveur[nom_outil]
    return {"ok": True, "outil": nom_outil, "spec": spec, "args_recus": arguments}


def appeler(catalogue_externe, nom_outil, arguments):
    """Cote CLIENT : Valmont appelle un outil chez un serveur MCP externe."""
    if nom_outil not in catalogue_externe:
        return {"erreur": "outil inconnu chez l'externe: " + nom_outil}
    spec = catalogue_externe[nom_outil]
    return {"ok": True, "outil": nom_outil, "spec": spec, "args_envoyes": arguments}


# Demonstration : un agent externe lit un manuscrit cote Valmont (sens expose)
r1 = servir(catalogue_expose, "valmont_get_manuscripts", {"status": "pending"})
print("SERVEUR (agent -> Valmont) : valmont_get_manuscripts")
print("  -> " + str(r1["spec"]))

# Demonstration : Valmont enrichit une fiche livre cote bookgraph (sens consomme)
r2 = appeler(catalogue_consomme, "bookgraph_enrich_book", {"book_id": 4892})
print()
print("CLIENT (Valmont -> bookgraph) : bookgraph_enrich_book")
print("  -> " + str(r2["spec"]))


SERVEUR (agent -> Valmont) : valmont_get_manuscripts
  -> {'verbe': 'lire', 'cible': 'manuscrit'}

CLIENT (Valmont -> bookgraph) : bookgraph_enrich_book
  -> {'verbe': 'enrichir', 'cible': 'livre'}


Deux appels, deux sens, deux configurations. **Rien ne dit à l'agent que
ces deux mondes existent** — c'est la conception du catalogue qui décide ce
qu'un agent peut atteindre. C'est pour ça que les deux sens se configurent
séparément : ils répondent à deux questions différentes (*que peut-on faire de
puis l'extérieur ?* vs *de quoi le site a-t-il besoin depuis l'extérieur ?*).


## 4. Le piège — le chevauchement fonctionnel

Voilà le cœur du problème. Valmont expose `valmont_get_book` (lire un livre)
**et** consomme `bookgraph_get_book_info` (lire un livre). Les deux font la
même chose, vue du modèle. Si un chatbot interne de Valmont a accès aux deux
catalogues (le consommé par construction, l'exposé s'il est branché en
boucle), comment choisit-il ?

Pour mesurer le chevauchement, on normalise chaque outil en une
**signature** `(verbe, cible)` et on calcule le recouvrement entre les deux
catalogues.


In [3]:
def signature(spec):
    """Normalise un outil en (verbe, cible) -- la signature fonctionnelle."""
    return (spec["verbe"], spec["cible"])


def signatures_catalogue(catalogue):
    """Ensemble des signatures presentes dans un catalogue."""
    return {signature(spec) for spec in catalogue.values()}


sig_expose = signatures_catalogue(catalogue_expose)
sig_consomme = signatures_catalogue(catalogue_consomme)

print("Signatures EXPOSE   : " + str(sorted(sig_expose)))
print()
print("Signatures CONSOMME : " + str(sorted(sig_consomme)))
print()
commun = sig_expose & sig_consomme
print("Chevauchement (present des deux cotes) : " + str(sorted(commun)))
print("  -> " + str(len(commun)) + " signature(s) commune(s)")


Signatures EXPOSE   : [('assigner', 'lecteur'), ('chercher', 'livre'), ('lire', 'compte-rendu'), ('lire', 'livre'), ('lire', 'manuscrit'), ('modifier', 'livre'), ('soumettre', 'manuscrit')]

Signatures CONSOMME : [('chercher', 'livre'), ('enrichir', 'livre'), ('lier', 'livre'), ('lire', 'auteur'), ('lire', 'livre'), ('modifier', 'livre'), ('soumettre', 'compte-rendu')]

Chevauchement (present des deux cotes) : [('chercher', 'livre'), ('lire', 'livre'), ('modifier', 'livre')]
  -> 3 signature(s) commune(s)


### Lecture du chevauchement

Les signatures `(lire, livre)` et `(chercher, livre)` sont présentes **des deux
côtés**. Concrètement : un chatbot Valmont qui veut lire une fiche livre a deux
chemins — `valmont_get_book` (interne) ou `bookgraph_get_book_info` (externe).
Lequel choisir ? S'ils divergent (données différentes, fraîcheur différente),
l'agent peut donner une réponse incohérente selon le chemin. S'ils
convergent, l'un des deux est **redondant**.


## 5. Mesurer — indice de Jaccard et redondance

L'indice de Jaccard entre les deux ensembles de signatures mesure à quel
point les catalogues se recouvrent : `|intersection| / |union|`. Un indice
élevé signifie que brancher le serveur externe ajoute peu de capacité
nouvelle — la plupart de ses outils dupliquent l'interne.


In [4]:
def jaccard(a, b):
    """Indice de Jaccard entre deux ensembles : |A n B| / |A u B|."""
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union else 0.0


j = jaccard(sig_expose, sig_consomme)
print("Indice de Jaccard (expose vs consomme) : " + format(j, ".2f"))
print("  |intersection| = " + str(len(sig_expose & sig_consomme)))
print("  |union|        = " + str(len(sig_expose | sig_consomme)))
print()
if j >= 0.4:
    print(">>> CHEVAUCHEMENT ELEVE : brancher ce serveur externe ajoute peu")
    print("    de capacite nouvelle. Plus de la moitie des signatures sont")
    print("    deja couvertes par le catalogue interne.")
else:
    print(">>> chevauchement faible : le serveur externe apporte majoritairement")
    print("    des signatures que le catalogue interne n'a pas.")


Indice de Jaccard (expose vs consomme) : 0.27
  |intersection| = 3
  |union|        = 11

>>> chevauchement faible : le serveur externe apporte majoritairement
    des signatures que le catalogue interne n'a pas.


### Interprétation

Un Jaccard élevé n'est pas forcément un défaut : il peut traduire une
**redondance volontaire** (un backup, une source de validation croisée). Mais
il pose la question opérationnelle : *pour chaque signature commune, quel est
l'outil canonique ?* Sans règle explicite, l'agent choisit au hasard — et la
cohérence des réponses s'en ressent.

La règle pratique : **un seul chemin par verbe métier**. Si deux outils font
`(lire, livre)`, l'un doit être marqué comme canonique et l'autre comme
dégénérescence (ou supprimé du catalogue branché).


## 6. Les outils externes non couverts — la vraie valeur du branchement

Le miroir du chevauchement : les signatures du catalogue consommé **absentes**
du catalogue interne. Ce sont elles que le branchement apporte de neuf.


In [5]:
apporte_neuf = sig_consomme - sig_expose
print("Signatures APPORTEES par le serveur externe (absentes de l'interne) :")
for s in sorted(apporte_neuf):
    # Retrouver le nom d'outil concret cote consomme pour cette signature
    noms = [n for n, spec in catalogue_consomme.items() if signature(spec) == s]
    print("  " + str(s) + "  via " + ", ".join(noms))
print()
print(">>> " + str(len(apporte_neuf)) + " signature(s) reellement nouvelle(s).")
print("    C'est la valeur reelle du branchement, derriere le bruit du chevauchement.")


Signatures APPORTEES par le serveur externe (absentes de l'interne) :
  ('enrichir', 'livre')  via bookgraph_enrich_book
  ('lier', 'livre')  via bookgraph_link_similar
  ('lire', 'auteur')  via bookgraph_get_author
  ('soumettre', 'compte-rendu')  via bookgraph_submit_review

>>> 4 signature(s) reellement nouvelle(s).
    C'est la valeur reelle du branchement, derriere le bruit du chevauchement.


### Lecture

Le verdict est en deux temps. **Au niveau global**, le chevauchement est
faible (Jaccard ~0,27) : sur 7 outils consommés, 4 apportent une signature que
l'interne n'a pas (`(enrichir, livre)`, `(lier, livre)`, `(lire, auteur)`,
`(soumettre, compte-rendu)`). Le branchement est donc justifié — il élargit
vraiment le champ fonctionnel.

**Mais un chevauchement global faible ne dispense pas d'examiner le
sous-ensemble écriture.** Parmi les 3 signatures communes, l'une est
`(modifier, livre)` — présente côté exposé (`valmont_update_book_meta`) **et**
côté consommé (`bookgraph_update_book`). C'est un **double-écrit** : si
l'agent met à jour la même fiche par les deux chemins, les deux sources
divergent. Un seul point de chevauchement sur un verbe d'écriture suffit à
créer un risque, même noyé dans un catalogue par ailleurs peu redondant.
C'est l'objet de l'exercice 2.

**C'est la question que ce notebook formalise** : *faut-il brancher ce serveur
MCP externe ?* se répond en deux dénombrements — global (le chevauchement
total dit si le branchement vaut le coup) et fin (le chevauchement sur les
verbes d'écriture dit où est le risque résiduel).


## 7. Exercices

Les trois exercices suivants manipulent les deux catalogues synthétiques.
Les stub sont à compléter — `return None` ou `pass`.


### Exercice 1 — la redondance par outil

Écrire une fonction qui, pour chaque outil du catalogue consommé, indique s'il
est **redondant** (sa signature existe déjà dans le catalogue exposé) ou
**nouveau**. Renvoie deux listes : les noms d'outils redondants et les noms
d'outils nouveaux.


In [6]:
def partitionner_redondance(expose, consomme):
    """Renvoie (redondants, nouveaux) -- listes de noms d'outils du consomme.

    Un outil consomme est redondant si sa signature existe deja cote expose.
    """
    # TODO : utiliser signature() et signatures_catalogue().
    return None, None


### Exercice 2 — le coût d'un double-écrit

Un outil redondant peut causer un **double-écrit** : si l'agent met à jour une
fiche livre via `valmont_update_book_meta` (interne) puis via un outil externe
équivalent, les deux sources divergent. Écrire une fonction
`detecter_double_ecrit` qui renvoie les signatures `(verbe, cible)` présentes
des deux côtés **avec un verbe d'écriture** (`modifier`, `soumettre`,
`enrichir`, `lier`…), c'est-à-dire le sous-ensemble du chevauchement qui est
dangereux (lecture redondante = tolérable, écriture redondante = risque).


In [7]:
VERBES_ECRITURE = {"modifier", "soumettre", "enrichir", "lier", "assigner", "creer", "supprimer"}

def detecter_double_ecrit(expose, consomme):
    """Renvoie les signatures (verbe, cible) presentes des deux cotes
    ET dont le verbe est un verbe d'ecriture -- le sous-ensemble dangereux
    du chevauchement.
    """
    # TODO : croiser signatures et VERBES_ECRITURE.
    return None


### Exercice 3 — choisir le canonical

Pour chaque signature commune, il faut désigner un **canonical** (l'outil à
utiliser par défaut). Écrire une fonction `choisir_canonical` qui, étant donné
les deux catalogues et une signature commune, renvoie le nom d'outil
canonique selon une règle simple : *préférer l'outil interne* (préfixe
`valmont_`) sauf si l'externe est le seul à offrir la signature.


In [8]:
def choisir_canonical(expose, consomme, sig_commune):
    """Pour une signature commune, renvoie le nom d'outil canonique.

    Regle : preferer l'outil interne (valmont_*); si aucun interne n'offre
    cette signature, prendre l'externe.
    """
    # TODO : parcourir expose puis consomme pour trouver l'outil de signature donnee.
    return None


## 8. Provenance et limites

**Ce que ce notebook mesure.** La *structure* d'un chevauchement
cross-catalogue : étant donné un catalogue exposé et un catalogue consommé,
combien d'outils se dupliquent, combien apportent du neuf, et quel
sous-ensemble du chevauchement est dangereux (écriture). Tout est déterministe
sur fixture synthétique.

**Ce qu'il ne mesure pas.** La qualité réelle des outils (un outil externe
peut avoir la même signature qu'un interne et renvoyer des données
radicalement différentes), la latence, le coût. La signature `(verbe, cible)`
est une **proxy** : elle dit que deux outils *font la même chose* au niveau
taxonomique, pas qu'ils renvoient le même résultat.

**La limite de la proxy.** Deux outils `(lire, livre)` peuvent l'un lire dans
la base WordPress et l'autre chez un agrégateur distant — même verbe, même
cible, données différentes. Le chevauchement mesuré ici est donc un **signal
d'alarme**, pas un verdict : il désigne les paires à examiner, pas les paires
à supprimer.

**Pour aller plus loin.**
- [`auditer-un-serveur-mcp.ipynb`](auditer-un-serveur-mcp.ipynb) — l'autre
  moitié : classifier les outils d'un **seul** catalogue en CRUD générique vs
  verbe métier (ce notebook compare **deux** catalogues).
- [`livresagites-parcours.md`](livresagites-parcours.md) Parcours 0 — la
  confusion fréquente entre les deux sens du protocole, dont ce notebook est
  l'illustration exécutable.
